# 8-1 실습 예제 · Airflow 서비스 구축

**데이터 엔지니어링 & RAG 파이프라인 · 8주차 1일차** — 수업에서 `docker compose up -d` 한 줄로 띄운 일곱 컨테이너가 **각각 무슨 일을 하는지**, 하나씩 켜고 꺼 보며 확인합니다.

- 걸리는 시간: 약 30분
- 키: 필요 없습니다(전부 키 없이 끝까지 돕니다).

### 시작하는 방법
1. 「Drive로 복사」를 누른 뒤 위에서부터 한 칸씩 실행합니다(칸 왼쪽 ▶ 또는 Shift+Enter).
2. 준비 칸 둘(설치 · 도구)은 처음 한 번만 실행합니다. 설치는 1~2분 걸립니다.
3. 예제마다 **확인 포인트**와 **바꿔 보기**가 있습니다. 확인 포인트의 값이 화면과 같은지 보고, 바꿔 보기는 직접 고쳐 다시 실행합니다.

### 수업 환경과 다른 점
수업에서는 `docker compose` 로 일곱 컨테이너를 띄웠습니다. 여기서는 같은 판(Airflow 3.3.2)을 코랩 안 가상환경 하나에 넣어, 설치 걱정 없이 DAG 에만 집중합니다. 코랩 런타임이 끊기면 처음부터 다시 실행하면 됩니다.

### 예제 구성
| 번호 | 예제 | 수업 노트와 이어지는 곳 |
|---|---|---|
| 1 | 설치된 것 살펴보기 — 설정과 메타데이터 DB | 실습 4 · 5단계(환경변수 · DB 초기화) |
| 2 | 서비스 켜기 — 구성 요소와 상태 점검 | 이론 「Airflow의 구성 요소」 · 실습 6단계 |
| 3 | 첫 DAG 를 스케줄러에 맡기기 | UI·CLI 탐색 6단계(예제 DAG 돌리기) |
| 4 | 스케줄러만 일부러 끄기 | 실습 8단계(일부러 내렸다가 다시 올리기) |
| 5 | 웹 UI 가 부르는 REST API 직접 부르기 | UI·CLI 탐색 4단계(CLI 로 같은 것 확인) |

## 준비

In [ ]:
# 준비 1 — Airflow 3.3.2 설치 (1~2분, 처음 한 번만)
import os, subprocess, sys, time

os.environ["TZ"] = "Asia/Seoul"          # 코랩 시계는 UTC — 화면의 시각을 한국 기준으로
time.tzset()

AF_VER = "3.3.2"                          # 수업과 같은 판
PY = f"{sys.version_info.major}.{sys.version_info.minor}"
HOME = "/content"
VENV = f"{HOME}/af"                       # 코랩 꾸러미와 섞이지 않게 가상환경에 설치합니다
PORT = 18080                              # 수업은 8080 — 코랩은 8080 을 이미 쓰고 있어 다른 번호를 씁니다

os.environ.update({
    "AIRFLOW_HOME": f"{HOME}/airflow",
    "AIRFLOW__CORE__DAGS_FOLDER": f"{HOME}/dags",      # 수업의 ~/airflow-lab/dags 와 같은 자리
    "AIRFLOW__CORE__LOAD_EXAMPLES": "False",
    "AIRFLOW__CORE__PARALLELISM": "4",
    "AIRFLOW__API__PORT": str(PORT),
    "AIRFLOW__API__BASE_URL": f"http://localhost:{PORT}",
    "AIRFLOW__CORE__EXECUTION_API_SERVER_URL": f"http://localhost:{PORT}/execution/",
    "NO_COLOR": "1",                                    # 로그에 색 코드가 섞이지 않게
    "PATH": f"{VENV}/bin:" + os.environ["PATH"],
})
for d in ("dags", "data"):
    os.makedirs(f"{HOME}/{d}", exist_ok=True)

if not os.path.exists(f"{VENV}/bin/airflow"):
    print("설치 중입니다 …")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True, capture_output=True)
    subprocess.run([sys.executable, "-m", "uv", "venv", "-q", VENV, "--python", sys.executable], check=True)
    r = subprocess.run(
        [sys.executable, "-m", "uv", "pip", "install", "-q", "--python", f"{VENV}/bin/python",
         f"apache-airflow=={AF_VER}", "graphviz",
         "--constraint", f"https://raw.githubusercontent.com/apache/airflow/constraints-{AF_VER}/constraints-{PY}.txt"],
        capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])
        raise SystemExit("설치하지 못했습니다. 이 칸을 한 번 더 실행해 주세요.")

r = subprocess.run("airflow db migrate", shell=True, capture_output=True, text=True)   # 메타데이터 DB(SQLite) 만들기
v = subprocess.run("airflow version", shell=True, capture_output=True, text=True).stdout.strip().splitlines()[-1]
print("설치 확인 · Airflow", v, "· 파이썬", PY)
print("DAG 폴더 ·", os.environ["AIRFLOW__CORE__DAGS_FOLDER"])

In [ ]:
# 준비 2 — 이 노트북에서 쓰는 도구 셋 (처음 한 번만)
import json, re
from datetime import datetime, timezone
from IPython.display import Image, display

_LOG = re.compile(r"^\d{4}-\d\d-\d\dT\S+ \[")
_ANSI = re.compile(r"\x1b\[[0-9;]*m")


def _sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return _ANSI.sub("", r.stdout + r.stderr)
_NOISE = ("graphviz", "You might need to install", "Please be reminded", "DagRun failed")


def af(args, show=True):
    """수업의 af 함수와 같습니다 — af("dags list") = airflow dags list. 로그 줄은 빼고 결과만 보여 줍니다."""
    keep = [l for l in _sh("airflow " + args).splitlines()
            if l.strip() and not _LOG.match(l) and not any(n in l for n in _NOISE)]
    text = "\n".join(keep)
    if show:
        print(text)
        return None                         # 화면에 찍었으면 돌려주지 않습니다(칸 아래에 한 번 더 나오지 않게)
    return text


def graph(dag_id):
    """DAG 모양을 그림으로 — 웹 UI 의 Graph 화면과 같은 것입니다."""
    png = f"{HOME}/data/{dag_id}.png"
    if os.path.exists(png):
        os.remove(png)
    af(f"dags show {dag_id} --save {png}", show=False)
    if not os.path.exists(png):             # 아직 등록 전인 새 파일이면 먼저 읽게 한 뒤 다시
        af("dags reserialize", show=False)
        af(f"dags show {dag_id} --save {png}", show=False)
    if os.path.exists(png):
        display(Image(png))
    else:                                   # 그림 도구가 없으면 화살표만 글자로
        print("\n".join(l.strip() for l in af(f"dags show {dag_id}", show=False).splitlines() if "->" in l))


def _utc(s):
    return datetime.fromisoformat(s).replace(tzinfo=timezone.utc)      # DB 의 시각은 UTC


def _kst(s):
    return _utc(s).astimezone().strftime("%H:%M:%S.%f")[:-3] if s else "-"


def run_dag(dag_id, conf=None):
    """DAG 를 한 번 끝까지 돌리고(airflow dags test), 태스크가 찍은 글과 태스크별 상태를 보여 줍니다."""
    cmd = f"airflow dags test {dag_id}" + (f" -c '{json.dumps(conf)}'" if conf else "")
    print(f"$ {cmd}\n실행 중입니다 …\n")
    out = _sh(cmd)
    task, tb = "-", []

    def flush():
        if tb:
            print(f"  [{task}] 오류 → {tb[-1].strip()}")
            tb.clear()

    for l in out.splitlines():
        m = re.search(r"\[DAG TEST\] starting task_id=(\S+)", l)
        if m:
            flush()
            task = m.group(1)
            continue
        if _LOG.match(l):
            flush()
            m = re.search(r"Done\. Returned value was: (.*?)\s+\[airflow", l)
            if m and m.group(1) != "None":
                print(f"  [{task}] 돌려준 값(XCom) → {m.group(1)}")
            m = re.search(r"Branch into (\S+)", l)
            if m:
                print(f"  [{task}] 고른 갈래 → {m.group(1)}")
            if "Skipping downstream tasks" in l and "ShortCircuitOperator" in l:
                print(f"  [{task}] False 를 돌려줘 뒤 태스크를 건너뜁니다")
            continue
        if not l.strip() or any(n in l for n in _NOISE):
            continue
        if tb or l.startswith("Traceback"):
            tb.append(l)
        else:
            print(f"  [{task}] {l}")
    flush()

    m = re.search(r"run_id=(manual__[^\s,]+)", out)
    if not m:                               # DAG 를 읽지 못한 경우 — 원인을 그대로 보여 줍니다
        print("\n".join(l for l in out.splitlines() if not _LOG.match(l))[-1500:])
        print("\nDAG 를 돌리지 못했습니다. 위 오류와 af(\"dags list-import-errors\") 를 확인해 주세요.")
        return {}
    import sqlite3                           # 태스크 상태는 메타데이터 DB 에서 바로 읽습니다(웹 UI 의 Grid 가 보는 곳)
    db = sqlite3.connect(f"{HOME}/airflow/airflow.db")
    rows = [dict(zip(("task_id", "state", "start_date", "end_date", "try_number"), r)) for r in db.execute(
        "select task_id, state, start_date, end_date, try_number from task_instance where dag_id=? and run_id=?",
        (dag_id, m.group(1)))]
    db.close()
    rows.sort(key=lambda x: x.get("start_date") or "9")
    print(f"\n{'태스크':<16}{'상태':<18}{'시도':<5}{'시작(한국 시각)':<16}걸린 시간")
    for x in rows:
        sec = ""
        if x.get("start_date") and x.get("end_date"):
            sec = f"{(_utc(x['end_date']) - _utc(x['start_date'])).total_seconds():.2f}초"
        print(f"{x['task_id']:<19}{(x['state'] or '-'):<20}{x['try_number']:<7}{_kst(x.get('start_date')):<20}{sec}")
    state = re.search(r"DagRun Finished:.*?state=(\w+)", out)
    print(f"\n실행 전체 상태 → {state.group(1) if state else '?'}")
    return {x["task_id"]: x["state"] for x in rows}


print("도구 준비 완료 · af( ) · graph( ) · run_dag( )")

In [ ]:
def start_airflow():
    """스케줄러까지 켭니다(airflow standalone) — 수업의 docker compose up -d 에 해당합니다. 20~40초 걸립니다."""
    import urllib.request
    global _SA
    if "_SA" in globals() and _SA.poll() is None:
        print("이미 켜져 있습니다")
        return
    _SA = subprocess.Popen("exec airflow standalone > /content/data/standalone.log 2>&1", shell=True,
                           start_new_session=True)      # 딸린 프로세스까지 한 묶음으로
    for _ in range(90):
        try:
            h = json.loads(urllib.request.urlopen(f"http://localhost:{PORT}/api/v2/monitor/health", timeout=3).read())
            if h["scheduler"]["status"] == "healthy" and h["dag_processor"]["status"] == "healthy":
                print("켜졌습니다 ·", " · ".join(f"{k} {v['status']}" for k, v in h.items()))
                return
        except Exception:
            pass
        time.sleep(2)
    print("3분 안에 켜지지 않았습니다. 로그의 끝부분입니다 ↓")
    print("\n".join(_ANSI.sub("", open("/content/data/standalone.log").read()).splitlines()[-15:]))


def stop_airflow():
    """스케줄러를 끕니다 — 수업의 docker compose stop 에 해당합니다."""
    global _SA
    if "_SA" in globals() and _SA.poll() is None:
        import signal
        os.killpg(_SA.pid, signal.SIGTERM)
        try:
            _SA.wait(timeout=30)
        except Exception:
            os.killpg(_SA.pid, signal.SIGKILL)
    print("껐습니다")


print("도구 준비 완료 · start_airflow( ) · stop_airflow( )")

## 예제 1 · 설치된 것 살펴보기 — 설정과 메타데이터 DB

Airflow 는 「지금 무엇이 어떤 상태인지」를 전부 **메타데이터 DB** 에 적어 둡니다. 수업 환경에서는 `postgres` 컨테이너가 그 일을 했고, 여기서는 파일 하나(SQLite)가 합니다.
준비 칸의 `airflow db migrate` 가 수업 5단계의 `docker compose up airflow-init` 과 같은 일(표 만들기)을 이미 해 두었습니다.

In [ ]:
print("판            ", af("version", show=False))
print("실행기         ", af("config get-value core executor", show=False))
print("DAG 폴더       ", af("config get-value core dags_folder", show=False))
print("예제 DAG 싣기  ", af("config get-value core load_examples", show=False))
print("DB 주소        ", af("config get-value database sql_alchemy_conn", show=False))

import sqlite3
db = sqlite3.connect(f"{HOME}/airflow/airflow.db")
tables = [r[0] for r in db.execute("select name from sqlite_master where type='table' order by name")]
db.close()
print(f"\n메타데이터 DB 의 표 {len(tables)}개 — 이름에 dag · task 가 든 것:")
print("  " + " · ".join(t for t in tables if t.startswith(("dag", "task"))))

***확인 포인트***: 실행기는 `LocalExecutor`, DB 주소는 `sqlite:////content/airflow/airflow.db` 입니다. 표가 수십 개 있고 그중 `dag` · `dag_run` · `task_instance` 가 보입니다.
웹 UI 의 Dags 목록 · Grid 의 색칠된 칸은 전부 이 표들을 읽어 그린 것입니다.

| | 수업 환경(docker compose) | 이 노트북 |
|---|---|---|
| 메타데이터 DB | `postgres` 컨테이너 | SQLite 파일 하나 |
| 실행기 | CeleryExecutor(`redis` + `airflow-worker`) | LocalExecutor(스케줄러가 직접 일꾼을 띄움) |

**바꿔 보기**
- `af("config get-value core parallelism")` 으로 한꺼번에 돌릴 수 있는 태스크 수를 확인합니다. 준비 칸에서 어떤 환경변수로 이 값을 정했는지 찾아봅니다(`AIRFLOW__구역__이름` 규칙 — 수업의 `.env` 와 같은 방식입니다).

## 예제 2 · 서비스 켜기 — 구성 요소와 상태 점검

이제 서비스를 켭니다. `start_airflow()` 는 `airflow standalone` 을 뒤에서 띄우고, 수업에서 쓴 것과 같은 상태 점검 주소(`/api/v2/monitor/health`)가 healthy 를 줄 때까지 기다립니다.

In [ ]:
start_airflow()

import urllib.request
health = json.loads(urllib.request.urlopen(f"http://localhost:{PORT}/api/v2/monitor/health").read())
print()
print(json.dumps(health, indent=2))

In [ ]:
# 떠 있는 프로세스를 구성 요소별로 셉니다 — 수업의 docker compose ps 에 해당합니다
ps = subprocess.run("ps -eo args", shell=True, capture_output=True, text=True).stdout.splitlines()
parts = {
    "api-server (웹 UI · REST API)": "api_server",
    "scheduler (실행을 만들고 태스크를 배정)": "airflow scheduler",
    "dag-processor (dags/ 의 파일을 읽어 등록)": "airflow dag-processor",
    "triggerer (기다리는 태스크 담당)": "airflow triggerer",
    "worker (태스크를 실제로 실행)": "airflow worker",
}
for label, key in parts.items():
    n = sum(1 for l in ps if key in l)
    print(f"{label:42s} 프로세스 {n}개")

***확인 포인트***: health 의 네 항목(metadatabase · scheduler · triggerer · dag_processor)이 모두 `healthy` 이고, 스케줄러 · triggerer · dag_processor 에는 **마지막 심장박동 시각**(heartbeat)이 찍혀 있습니다.
프로세스 표에서 다섯 구성 요소가 모두 1개 이상입니다(worker 는 `PARALLELISM` 값만큼 미리 떠 있습니다).

수업의 일곱 컨테이너와 이렇게 대응합니다.

| 수업의 컨테이너 | 여기서는 | 하는 일 |
|---|---|---|
| `postgres` | SQLite 파일 | 상태를 전부 기록 |
| `redis` | 없음 | 스케줄러 → 워커로 일감을 넘기는 줄(LocalExecutor 는 필요 없음) |
| `airflow-apiserver` | api-server | 웹 UI 와 REST API, 태스크가 상태를 보고하는 창구 |
| `airflow-scheduler` | scheduler | 스케줄을 보고 실행을 만들고 태스크를 배정 |
| `airflow-dag-processor` | dag-processor | `dags/` 폴더의 파이썬 파일을 읽어 DB 에 등록 |
| `airflow-worker` | worker(스케줄러가 띄움) | 태스크의 파이썬 함수를 실제로 실행 |
| `airflow-triggerer` | triggerer | 「무엇을 기다리는」 태스크를 대신 기다림 |

**바꿔 보기**
- 위 health 칸을 10초 뒤에 다시 실행합니다. heartbeat 시각이 바뀌었나요? 이 시각이 오래 멈춰 있으면 Airflow 는 그 구성 요소를 unhealthy 로 봅니다.

## 예제 3 · 첫 DAG 를 스케줄러에 맡기기

파일을 `dags/` 에 넣고 → 등록되는지 보고 → 켜고 → 실행을 요청하고 → 끝날 때까지 지켜봅니다. 수업에서 예제 DAG 로 한 흐름 그대로이고, 8-2 에서 직접 쓸 DAG 의 가장 작은 모양입니다.

In [ ]:
%%writefile /content/dags/hello_service.py
# hello_service.py — 서비스가 도는지 확인하는 가장 작은 DAG
from datetime import datetime

from airflow.sdk import DAG
from airflow.providers.standard.operators.python import PythonOperator


def say_hello():
    print("hello, airflow service!")


with DAG(
    dag_id="hello_service",
    start_date=datetime(2026, 1, 1),
    schedule=None,                          # 요청할 때만 돕니다
    catchup=False,
    tags=["week8"],
) as dag:
    PythonOperator(task_id="say_hello", python_callable=say_hello)

In [ ]:
af("dags reserialize", show=False)          # dag-processor 가 훑기를 기다리지 않고 지금 읽게 합니다
af("dags list")

In [ ]:
def wait_run(dag_id, seconds=60):
    """가장 최근 실행이 끝날 때까지 5초마다 상태를 찍습니다."""
    for i in range(seconds // 5):
        runs = json.loads(af(f"dags list-runs {dag_id} -o json", show=False) or "[]")
        state = runs[0]["state"] if runs else "(아직 없음)"
        print(f"{i * 5:>3}초  {state}")
        if state in ("success", "failed"):
            return state
        time.sleep(5)
    return state


af("dags unpause hello_service", show=False)    # 새 DAG 는 일시정지 상태로 등록됩니다
af("dags trigger hello_service", show=False)
state = wait_run("hello_service")

***확인 포인트***: `dags list` 에 `hello_service` 가 보이고(`is_paused` 가 True), trigger 뒤 상태가 `queued` 나 `running` 을 거쳐 `success` 로 끝납니다(빠르면 첫 줄이 이미 running 입니다).

이 한 번의 실행에 구성 요소가 전부 관여했습니다. dag-processor 가 파일을 읽어 등록했고, 스케줄러가 실행을 만들어 워커에 넘겼고, 워커가 함수를 실행해 api-server 로 결과를 보고했고, 그 기록이 메타데이터 DB 에 남았습니다.

**바꿔 보기**
- `af("dags unpause …")` 줄을 빼고(먼저 `af("dags pause hello_service")` 로 다시 멈춘 뒤) trigger 만 하면 상태가 어디에 머무나요? 수업에서 「unpause 를 빼면 queued 에 머문다」고 한 것을 직접 봅니다.

## 예제 4 · 스케줄러만 일부러 끄기

수업 8단계에서는 전체를 내렸다가 올렸습니다. 여기서는 **스케줄러 하나만** 끕니다. 나머지(웹 UI · DB)는 살아 있는데 무슨 일이 생기는지 봅니다.

In [ ]:
import signal

pid = int(subprocess.run("pgrep -of 'airflow scheduler'", shell=True, capture_output=True, text=True).stdout.split()[0])
os.kill(pid, signal.SIGSTOP)                # 스케줄러를 얼립니다(죽이지 않고 멈춰 세움)
print("스케줄러(pid", pid, ")를 멈췄습니다\n")

af("dags trigger hello_service", show=False)
print("스케줄러가 멈춘 채로 실행을 요청했습니다 — 20초 지켜봅니다")
state = wait_run("hello_service", seconds=20)

In [ ]:
print("스케줄러의 심장박동이 끊긴 지 40초가 넘을 때까지 기다립니다 …")
time.sleep(25)
health = json.loads(urllib.request.urlopen(f"http://localhost:{PORT}/api/v2/monitor/health").read())
for k, v in health.items():
    print(f"{k:14s} {v['status']}")

In [ ]:
os.kill(pid, signal.SIGCONT)                # 스케줄러를 다시 움직이게 합니다
print("스케줄러를 다시 움직였습니다\n")
state = wait_run("hello_service")

***확인 포인트***: 스케줄러가 멈춘 동안 실행은 **`queued` 에서 움직이지 않습니다**. 요청은 받아졌는데(api-server 는 살아 있으니) 일을 배정할 사람이 없는 것입니다.
health 에서는 **scheduler 만 `unhealthy`** 이고 나머지는 healthy 입니다. 스케줄러를 다시 움직이면 밀려 있던 실행이 곧 `success` 가 됩니다.

「웹 UI 는 열리는데 DAG 가 돌지 않는다」의 가장 흔한 원인이 이것입니다. 수업 환경이라면 `docker compose ps` 에서 scheduler 가 내려가 있거나 unhealthy 일 것이고, `docker compose logs airflow-scheduler` 로 이유를 봅니다.

**바꿔 보기**
- 같은 방법으로 `'airflow dag-processor'` 를 멈추고 새 DAG 파일을 넣어 봅니다. `af("dags list")` 에 새 DAG 가 나타나나요? (확인한 뒤 SIGCONT 로 되살립니다. `af("dags reserialize")` 는 dag-processor 없이도 직접 읽는다는 점에 주의합니다 — 이번에는 쓰지 않고 기다려 봅니다.)

## 예제 5 · 웹 UI 가 부르는 REST API 직접 부르기

웹 UI 의 화면은 전부 api-server 의 REST API 를 불러 그립니다. CLI(`af`)는 DB 를 직접 읽고, 웹 UI 와 외부 프로그램은 이 API 를 씁니다.
먼저 관리자 비밀번호로 **토큰**을 받고, 그 토큰으로 DAG 목록과 실행 이력을 받습니다.

In [ ]:
import urllib.request

BASE = f"http://localhost:{PORT}"
pw = json.load(open(f"{HOME}/airflow/simple_auth_manager_passwords.json.generated"))["admin"]   # standalone 이 만든 관리자 비밀번호


def call(path, data=None, token=None):
    req = urllib.request.Request(BASE + path, data=json.dumps(data).encode() if data else None,
                                 headers={"Content-Type": "application/json",
                                          **({"Authorization": "Bearer " + token} if token else {})})
    return json.loads(urllib.request.urlopen(req).read())


token = call("/auth/token", {"username": "admin", "password": pw})["access_token"]
print("토큰 받음 · 길이", len(token), "\n")

for d in call("/api/v2/dags", token=token)["dags"]:
    print(f"DAG   {d['dag_id']:16s} 일시정지 {d['is_paused']}")

for r in call("/api/v2/dags/hello_service/dagRuns", token=token)["dag_runs"]:
    print(f"실행  {r['dag_run_id'][:32]:34s} {r['state']}")

In [ ]:
# 토큰 없이 부르면 어떻게 되는지 봅니다
try:
    call("/api/v2/dags")
except Exception as e:
    print("토큰 없이 부른 결과 →", e)

***확인 포인트***: DAG 목록에 `hello_service` 가 있고, 실행 이력에 예제 3 · 4 에서 만든 실행 두 개가 `success` 로 나옵니다. 토큰 없이 부르면 **401 Unauthorized** 로 거절됩니다.
상태 점검 주소(`/api/v2/monitor/health`)만은 토큰 없이 열려 있습니다 — 감시 도구가 로그인 없이 부를 수 있어야 하기 때문입니다.

**바꿔 보기**
- 실행 이력 응답에는 다른 칸도 많습니다. `r.keys()` 를 찍어 보고 시작 · 끝 시각과 실행 종류(`run_type`)를 함께 출력해 봅니다. 웹 UI 의 실행 목록이 보여 주는 값들입니다.

## 정리 — 서비스 끄기
수업의 `docker compose stop` 에 해당합니다. DB 파일과 DAG 파일은 남아 있어 `start_airflow()` 로 다시 켜면 이력이 그대로 보입니다.

In [ ]:
stop_airflow()

## 마무리 — 결과 남기기 (선택)

- 가장 기억에 남는 예제 하나의 **상태 표 화면**을 캡처합니다.
- 「바꿔 보기」에서 직접 고친 것과, 고쳤더니 무엇이 달라졌는지 한두 줄로 적습니다.
- 패들릿 **「8주 · Orchestration · AI Service」** 칸에 `이름 — 8-1 실습 예제` 제목으로 올립니다.

수업 노트: https://data-eng.jobability.co.kr